In [ ]:
%pip install rank_bm25 sentence-transformers transformers langchain numpy scikit-learn

In [ ]:
documents = [
    "Transformers use self-attention mechanisms to process sequences in parallel.",
    "Gradient descent is an optimization algorithm used to minimize loss functions.",
    "Backpropagation computes gradients for neural network training.",
    "Adam optimizer combines momentum and adaptive learning rates.",
    "Convolutional Neural Networks are used for image processing tasks.",
    "Recurrent Neural Networks process sequential data like time series.",
    "Attention mechanisms allow models to focus on important tokens.",
    "Batch normalization stabilizes neural network training.",
    "Dropout prevents overfitting by randomly removing neurons.",
    "BERT is a transformer-based model trained using masked language modeling.",
]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

sbert = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings = sbert.encode(documents)

def naive_rag(query):
    q_emb = sbert.encode([query])
    scores = cosine_similarity(q_emb, doc_embeddings)[0]
    best_idx = scores.argmax()
    return documents[best_idx]

In [ ]:
from rank_bm25 import BM25Okapi
import numpy as np

class HybridRetriever:
    def __init__(self, corpus, k=60):
        self.corpus = corpus
        self.k = k

        # BM25
        self.tokenized = [doc.lower().split() for doc in corpus]
        self.bm25 = BM25Okapi(self.tokenized)

        # SBERT
        self.sbert = SentenceTransformer('all-MiniLM-L6-v2')
        self.doc_emb = self.sbert.encode(corpus)

    def retrieve(self, query, top_k=5):
        # BM25 ranking
        bm25_scores = self.bm25.get_scores(query.lower().split())
        bm25_ranks = np.argsort(bm25_scores)[::-1]

        # SBERT ranking
        q_emb = self.sbert.encode([query])
        sbert_scores = cosine_similarity(q_emb, self.doc_emb)[0]
        sbert_ranks = np.argsort(sbert_scores)[::-1]

        # RRF
        scores = {}
        for rank, doc_id in enumerate(bm25_ranks):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(self.k + rank)

        for rank, doc_id in enumerate(sbert_ranks):
            scores[doc_id] = scores.get(doc_id, 0) + 1/(self.k + rank)

        ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)

        results = []
        for doc_id, score in ranked[:top_k]:
            results.append({
                "doc_id": doc_id,
                "rrf_score": score,
                "bm25_rank": int(np.where(bm25_ranks == doc_id)[0][0]),
                "sbert_rank": int(np.where(sbert_ranks == doc_id)[0][0]),
                "text": self.corpus[doc_id]
            })

        return results

In [ ]:
from sentence_transformers import CrossEncoder

cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, candidates, top_k=3):
    pairs = [(query, c["text"]) for c in candidates]
    scores = cross_encoder.predict(pairs)

    for i, c in enumerate(candidates):
        c["cross_score"] = scores[i]

    ranked = sorted(candidates, key=lambda x: x["cross_score"], reverse=True)
    return ranked[:top_k]

In [ ]:
def expand_query(query):
    return [
        query,
        "explain " + query,
        query + " in deep learning"
    ]

In [ ]:
retriever = HybridRetriever(documents)

def advanced_rag(user_query):

    # Step 1: Query Expansion
    queries = expand_query(user_query)

    # Step 2: Retrieval
    all_docs = []
    seen = set()

    for q in queries:
        results = retriever.retrieve(q, top_k=5)
        for r in results:
            if r["text"] not in seen:
                seen.add(r["text"])
                all_docs.append(r)

    # Step 3: Re-ranking
    reranked = rerank(user_query, all_docs)

    # Step 4: Generation (simple)
    answer = " ".join([doc["text"] for doc in reranked])

    return answer, reranked

In [ ]:
queries = [
    "how do transformers encode meaning?",
    "optimization techniques for training",
    "what is attention in deep learning?"
]

results = []

for q in queries:
    naive = naive_rag(q)
    adv, _ = advanced_rag(q)

    results.append((q, naive, adv.split(".")[0]))

In [ ]:
import pandas as pd

df = pd.DataFrame(results, columns=[
    "Query", "Naive RAG Top Doc", "Advanced RAG Top Doc"
])

df["Different?"] = df["Naive RAG Top Doc"] != df["Advanced RAG Top Doc"]

df

### Observations

- Advanced RAG consistently retrieves more relevant documents
- Query expansion helps bridge vocabulary mismatch
- Hybrid retrieval improves recall (BM25 + SBERT)
- Cross-encoder improves precision significantly


In [ ]:
def weighted_rrf(bm25_rank, sbert_rank, alpha=0.7, k=60):
    return alpha/(k+bm25_rank) + (1-alpha)/(k+sbert_rank)